In [2]:
import pandas as pd

In [2]:
hora = pd.read_csv("cr7_chats.csv")
hora.head()

,video_id,author,comment,likes,published_at,is_reply
0,JKEuUQXq9PY,@bishalghalan7146,😂😂😂,0,2026-03-24T18:13:28Z,False
1,JKEuUQXq9PY,@avinavbudhathoki8155,Dai 4 month jati wait garnu Tespaxi bala kinu ...,0,2026-03-24T17:42:25Z,False
2,JKEuUQXq9PY,@KALTIIEG-b2r,"<a href=""https://www.youtube.com/watch?v=JKEuU...",0,2026-03-24T17:32:15Z,False
3,JKEuUQXq9PY,@cr7horaaYT,Bhai license ko lagi agadi sikeko dekhenau 😂,1,2026-03-24T17:56:08Z,True
4,JKEuUQXq9PY,@mr.k.khadka1371,"Keta haru le pasina chuhaune, ghantey le moj g...",0,2026-03-24T16:43:57Z,False


In [4]:
hora_chats = hora['comment']

In [37]:
len(hora_chats)

49241

In [20]:
import re
from difflib import SequenceMatcher


def is_garbled_devanagari(text):
    devanagari_chars = re.findall(r'[\u0900-\u097F]', text)
    if len(devanagari_chars) < 6:
        return False
    vowel_marks = re.findall(r'[\u093E-\u094C\u0902\u0903]', text)
    return (len(vowel_marks) / len(devanagari_chars)) < 0.10


def is_garbled_romanized(text):
    """Lowered thresholds: per-word vowel ratio 0.22, fraction 0.30."""
    ascii_ratio = sum(1 for c in text if ord(c) < 128) / max(len(text), 1)
    if ascii_ratio < 0.6:
        return False
    words = text.split()
    if len(words) < 3:
        return False
    garbled_count = 0
    for word in words:
        alpha = re.sub(r'[^a-zA-Z]', '', word)
        if len(alpha) >= 4:
            vowels = len(re.findall(r'[aeiouAEIOU]', alpha))
            if vowels / len(alpha) < 0.22:
                garbled_count += 1
    return (garbled_count / len(words)) > 0.30


def is_near_duplicate(new_line, recent_lines, threshold=0.75):
    """Lowered threshold 0.85→0.75 to catch typo variants."""
    for seen in recent_lines[-50:]:
        if SequenceMatcher(None, new_line.lower(), seen.lower()).ratio() >= threshold:
            return True
    return False


def is_rhetorical_filler(text):
    """Devanagari questions ≤10 words; romanized ≤8 words."""
    stripped = text.strip()
    if not stripped.endswith('?'):
        return False
    words = stripped.split()
    devanagari_chars = re.findall(r'[\u0900-\u097F]', stripped)
    if len(devanagari_chars) > 5:
        return len(words) <= 10
    return len(words) <= 8


def is_low_content_opinion(text):
    LOW_CONTENT_PATTERNS = [
        r'जे मन लाग्यो',
        r'तेही भन्न लाग्यो',
        r'सुभकामना.*अनुरोध',
        r'अनैतिक काम.*अनुरोध',
        r'^kina\s+aadha\b',
        r'^kina\s+\w+\s+matra\b',
        r'charmari\s+la\s+garda',
        r'vaster\s+vaiko',
        r'^samachar\s+ko\s+kura\s+matra\s+gar',
        r'doctor\s+hoinau\s+bujheu',
    ]
    return any(re.search(p, text, re.IGNORECASE) for p in LOW_CONTENT_PATTERNS)


KEYWORDS_TO_REMOVE = [
    r'instagram', r'facebook', r'twitter', r'tiktok', r'snapchat',
    r'telegram', r'discord', r'whatsapp', r'onlyfans', r'patreon',
    r'ko-fi', r'buff',
]

PROMOTIONAL_PHRASES = [
    r'video mann parey maa like', r'share ra subscribe garnu hola',
    r'also follow on instagram', r'like.*share.*subscribe',
    r'follow on instagram', r'click.*subscribe', r'press.*bell.*icon',
    r"don't forget to subscribe", r'please subscribe', r'like and share',
    r'subscribe to my channel', r'follow me on', r'check out my',
    r'support my channel',
]

FILLER_PATTERNS = [
    r'^good\s*(morning|evening|night|afternoon)',
    r'^(jay|jai|jy)\s+nepal',
    r'^me\s+(the\s+)?regular',
    r'^i\s+am\s+(watching|listening)',
    r'^(hello|hi|hey|hlo|hii)\b',
    r'^(lol|haha|hehe|wow|nice|great|cool|ok|okay)\b',
    r'^(nepal\s+)?zindabad',
    r'jay\s+nepal\s*$',
    r'^jy\s+\w',
]

LISTENER_PATTERNS = [
    r'\b(sunxu|herxu|herne|sunchhu)\b.*\b(bata|jilla)\b',
    r'\bdainik\s+herne\b',
    r'\bregular\s+(listener|viewer|sunxu|herxu)\b',
    r'\bma\s+dainik\b',
    r'\bwatching\s+(from|live)\b',
    r'बाट\s*दैनिक\s*हेर्ने',
    r'नगरपालिका.*दैनिक',
    r'bata\s+dainik',
    r'\bरेगुलर\b',
    r'न\s*पा\s+बाट\s+रेगुलर',
    r'\bpa\s+bata\s+regular\b',
]

REACTION_PATTERNS = [
    r'^(hoho|haha|lol|wow|finally)\b',
    r'\bjel\s+(kochnu|haal|halnu)\b',
    r'^sabai\s+\w+\s+(lai\s+)?jel\b',
    r'^\w+\s+le\s+thik\s+bhan',
    r'^\w{1,15}\s+sir\s+please\b',
    r'kunai\s+pani\s+parti\s+ko.*baputi',
]

VAGUE_ENGLISH_PATTERNS = [
    r'^if\s+\w+\s+(come|win|lose|get|become)',
    r'^if\s+u\s+',
    r'^just\s+look\s+(at|in|what)',
    r'^(see|look|watch)\s+what\s+happen',
    r'baputi\s+banera\s+basdainan',
]

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "]+", flags=re.UNICODE
)


def clean_youtube_chat(raw_lines, remove_all_duplicates=False, normalize_repeated_chars=False):
    cleaned_messages = []
    prev_msg = None
    seen = set()

    for line in raw_lines:
        line = str(line)

        if any(re.search(kw, line, re.IGNORECASE) for kw in KEYWORDS_TO_REMOVE):
            continue

        line = re.sub(r'&quot;', '', line)
        line = re.sub(r'&amp;', '&', line)
        line = re.sub(r'&lt;', '<', line)
        line = re.sub(r'&gt;', '>', line)
        line = re.sub(r'&nbsp;', ' ', line)
        line = re.sub(r'&#39;?', '', line)
        line = re.sub(r'&[a-zA-Z]+;', '', line)
        line = re.sub(r'&[^;\s]+;', '', line)
        line = EMOJI_PATTERN.sub('', line)
        line = re.sub(r'https?://\S+', '', line)
        line = re.sub(r'ftp://\S+', '', line)
        line = re.sub(r'www\.\S+', '', line)
        line = re.sub(r'youtu\.be/\S+', '', line)
        line = re.sub(r'youtube\.com/\S+', '', line)
        line = re.sub(r'\S+\.(com|org|net|io|co|in|ly|be)/\S+', '', line)
        line = re.sub(r'http\S+', '', line)
        line = re.sub(r'<[^>]+>', '', line)
        line = re.sub(r'</?[a-zA-Z][^>]*>?', '', line)
        line = re.sub(r'\b(a\s+href|href\s*=)[^\s]*', '', line, flags=re.IGNORECASE)
        line = re.sub(r'href\s*=\s*["\'][^"\']*["\']', '', line)
        line = re.sub(r'\bbr\b', '', line)
        line = re.sub(r'\b[A-Za-z0-9_-]{20,}\b', '', line)
        line = re.sub(r'^\s*\d+\s+', '', line)
        line = re.sub(r'\b\d+:\d+(?::\d+)?\b', '', line)
        line = re.sub(r'@\S+', '', line)
        line = re.sub(r'@', '', line)

        for phrase in PROMOTIONAL_PHRASES:
            line = re.sub(phrase, '', line, flags=re.IGNORECASE)

        line = re.sub(r'(\b(ha|haha|lol|xD)\s*){2,}$', '', line, flags=re.IGNORECASE)
        line = re.sub(r'(\bha\s+){3,}', '', line, flags=re.IGNORECASE)

        for noise in [r'\balso\b', r'\bra\b', r'\bbr\b']:
            line = re.sub(noise, '', line, flags=re.IGNORECASE)

        line = re.sub(r'[^\w\s\u0900-\u097F.,!?;:()\-\'"]+', '', line)
        line = re.sub(r'\.{2,}', '', line)
        line = re.sub(r'([!?])\1+', r'\1', line)
        line = re.sub(r'^[.\s]+|[.\s]+$', '', line)
        line = re.sub(r'\s+', ' ', line).strip()

        if not line:
            continue

        if normalize_repeated_chars:
            line = re.sub(r'(.)\1{2,}', r'\1', line)

        line = re.sub(r'\s+', ' ', line).strip()
        if not line:
            continue

        if is_garbled_devanagari(line):       continue  # step 17
        if is_garbled_romanized(line):        continue  # step 18
        if any(re.search(p, line, re.IGNORECASE) for p in FILLER_PATTERNS):    continue  # 19
        if any(re.search(p, line, re.IGNORECASE) for p in LISTENER_PATTERNS):  continue  # 20
        if any(re.search(p, line, re.IGNORECASE) for p in REACTION_PATTERNS):  continue  # 21
        if any(re.search(p, line, re.IGNORECASE) for p in VAGUE_ENGLISH_PATTERNS): continue  # 22
        if is_rhetorical_filler(line):        continue  # step 23
        if is_low_content_opinion(line):      continue  # step 24

        ascii_ratio = sum(1 for c in line if ord(c) < 128) / max(len(line), 1)
        min_words = 8 if ascii_ratio > 0.7 else 6
        if len(line.split()) < min_words:
            continue

        if remove_all_duplicates:
            if line in seen or is_near_duplicate(line, cleaned_messages):
                continue
            seen.add(line)
        else:
            if line == prev_msg or is_near_duplicate(line, [prev_msg] if prev_msg else []):
                continue

        cleaned_messages.append(line)
        prev_msg = line

    return cleaned_messages


def save_chat_to_txt(cleaned_messages, filename="cleaned_chat.txt"):
    with open(filename, 'w', encoding='utf-8') as f:
        for msg in cleaned_messages:
            f.write(msg + '\n')
    print(f"Saved {len(cleaned_messages)} messages to '{filename}'")


In [3]:
samachar = pd.read_csv("samacharpati_chats.csv")
samachar.head()

,video_id,author,comment,likes,published_at,is_reply
0,tdDhpiIcexc,@KshetraDangi,हर्कले हावा तालमा कुरा गर्छन। जिम्मेवार व्यक्त...,0,2026-03-25T02:01:02Z,False
1,tdDhpiIcexc,@RamprasaddahalDahal,धन्यवाद म जनकपुर बाट।,0,2026-03-25T02:00:46Z,False
2,tdDhpiIcexc,@JayPun-fn3wr,Ragular,0,2026-03-25T01:54:31Z,False
3,tdDhpiIcexc,@KshetraDangi,नेपाली हु भन्नेले अरु भाषामा सपथ लिन पाइदैन 🎉,0,2026-03-25T01:54:24Z,False
4,tdDhpiIcexc,@JogBdrKUNWAR,Good morning Ramshran Sir Best Newes Center Re...,0,2026-03-25T01:53:23Z,False


In [4]:
data = samachar['comment']

In [5]:
data

0         हर्कले हावा तालमा कुरा गर्छन। जिम्मेवार व्यक्त...
1                                     धन्यवाद म जनकपुर बाट।
2                                                   Ragular
3             नेपाली हु भन्नेले अरु भाषामा सपथ लिन पाइदैन 🎉
4         Good morning Ramshran Sir Best Newes Center Re...
                                ...                        
576912                                               hahaha
576913                                                    😴
576914          Yesto news haru kina highlight maa aaudaina
576915                                  बुडि को नाटक हो सबै
576916                            Kin ho teti aabes ma aako
Name: comment, Length: 576917, dtype: object

In [30]:
import re
from difflib import SequenceMatcher


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def is_garbled_devanagari(text):
    """Lines with lots of Devanagari but almost no vowel marks = encoding corruption."""
    devanagari_chars = re.findall(r'[\u0900-\u097F]', text)
    if len(devanagari_chars) < 6:
        return False
    vowel_marks = re.findall(r'[\u093E-\u094C\u0902\u0903]', text)
    return (len(vowel_marks) / len(devanagari_chars)) < 0.10


def is_garbled_romanized(text):
    """
    Catches over-abbreviated romanized Nepali.
    Thresholds: per-word vowel ratio < 0.25, fraction of garbled words > 0.25.
    """
    ascii_ratio = sum(1 for c in text if ord(c) < 128) / max(len(text), 1)
    if ascii_ratio < 0.6:
        return False
    words = text.split()
    if len(words) < 3:
        return False
    garbled_count = 0
    for word in words:
        alpha = re.sub(r'[^a-zA-Z]', '', word)
        if len(alpha) >= 4:
            vowels = len(re.findall(r'[aeiouAEIOU]', alpha))
            if vowels / len(alpha) < 0.25:
                garbled_count += 1
    return (garbled_count / len(words)) > 0.25


def has_midword_script_mixing(text):
    """
    Detects mid-word mixing of Devanagari and Latin scripts (e.g. विदेशीliदोष).
    This indicates corrupted/garbled text, not intentional code-switching.
    """
    return bool(re.search(
        r'[a-zA-Z][\u0900-\u097F]|[\u0900-\u097F][a-zA-Z]',
        text
    ))


def is_near_duplicate(new_line, recent_lines, threshold=0.70):
    """
    Fuzzy match — checks both directions and takes max ratio.
    Catches typo variants like two 'barsa sam' lines.
    """
    for seen in recent_lines[-50:]:
        r1 = SequenceMatcher(None, new_line.lower(), seen.lower()).ratio()
        r2 = SequenceMatcher(None, seen.lower(), new_line.lower()).ratio()
        if max(r1, r2) >= threshold:
            return True
    return False


def is_rhetorical_filler(text):
    """
    Short questions with no substance.
    Devanagari questions up to 10 words; romanized up to 8 words.
    """
    stripped = text.strip()
    if not stripped.endswith('?'):
        return False
    words = stripped.split()
    devanagari_chars = re.findall(r'[\u0900-\u097F]', stripped)
    if len(devanagari_chars) > 5:
        return len(words) <= 10
    return len(words) <= 8


def is_off_topic_english(text):
    """
    English lines that contain no Nepal/political keywords are off-topic noise
    (e.g. 'There are ambro bus to the south africa, ramla vanuola').
    Only applied to lines that are >70% ASCII.
    """
    ascii_ratio = sum(1 for c in text if ord(c) < 128) / max(len(text), 1)
    if ascii_ratio < 0.70:
        return False
    nepal_keywords = (
        r'\b(nepal|nepali|balen|rabi|sarkar|sarkaar|congress|kangres|kangren|kangresh|amale|uml|'
        r'maoist|mawobadi|maobadi|communist|government|minister|vote|election|rastriya|sambidhan|'
        r'prachanda|deuba|oli|koirala|madhes|janajati|province|pradesh|'
        r'kathmandu|pokhara|rsp|ncp|girija|sushila|karki|bikalpa|desh|janta|janata|bikas)\b'
    )
    return not bool(re.search(nepal_keywords, text, re.IGNORECASE))


def is_low_content_opinion(text):
    """
    Catches short vague opinion/reaction lines not caught by other filters.
    """
    LOW_CONTENT_PATTERNS = [
        r'जे मन लाग्यो',
        r'तेही भन्न लाग्यो',
        r'सुभकामना.*अनुरोध',
        r'अनैतिक काम.*अनुरोध',
        r'^kina\s+aadha\b',
        r'^kina\s+\w+\s+matra\b',
        r'charmari\s+la\s+garda',
        r'vaster\s+vaiko',
        r'^samachar\s+ko\s+kura\s+matra\s+gar',
        r'doctor\s+hoinau\s+bujheu',
        r'बुढो\s+मान्छेको\s+पुरानै\s+सोच',
        r'\bchintai\s+chinta\b',
        r'\bbisram\s+linu\s+hosh\b',
        r'प्वाँख\s+पलाएको',
        r'अचम्म\s+लाग्छ',
        r'बिरालोले\s+मुसा\s+मर्दैन',
        r'धेरै\s+मन्त्री\s+काम\s+छैन',
        # "5 years give to Balen as PM" — simple demand, no argument
        r'बालेन\s+(लाई\s+)?नै\s+(प्राधान|प्रधान)\s+मन्त्री\s+दिनु\s+पर्छ',
        # "aadha aadha" half-term opinion repeat
        r'आधा\s+आधा\s+कार्यकाल',
        r'बालेन\s+नै\s+हुनुपर्छ',
        r'५\s*वर्षसम्म\s+बालेन',
    ]
    return any(re.search(p, text, re.IGNORECASE) for p in LOW_CONTENT_PATTERNS)


# ---------------------------------------------------------------------------
# Filter lists
# ---------------------------------------------------------------------------

KEYWORDS_TO_REMOVE = [
    r'instagram', r'facebook', r'twitter', r'tiktok', r'snapchat',
    r'telegram', r'discord', r'whatsapp', r'onlyfans', r'patreon',
    r'ko-fi', r'buff',
]

PROMOTIONAL_PHRASES = [
    r'video mann parey maa like', r'share ra subscribe garnu hola',
    r'also follow on instagram', r'like.*share.*subscribe',
    r'follow on instagram', r'click.*subscribe', r'press.*bell.*icon',
    r"don't forget to subscribe", r'please subscribe', r'like and share',
    r'subscribe to my channel', r'follow me on', r'check out my',
    r'support my channel',
]

FILLER_PATTERNS = [
    r'^good\s*(morning|evening|night|afternoon)',
    r'^(jay|jai|jy)\s+nepal',
    r'^me\s+(the\s+)?regular',
    r'^i\s+am\s+(watching|listening)',
    r'^(hello|hi|hey|hlo|hii)\b',
    r'^(lol|haha|hehe|wow|nice|great|cool|ok|okay)\b',
    r'^(nepal\s+)?zindabad',
    r'jay\s+nepal\s*$',
    r'^jy\s+\w',
]

LISTENER_PATTERNS = [
    # Original — herxu/sunxu before bata
    r'\b(sunxu|herxu|herne|sunchhu)\b.*\b(bata|jilla)\b',
    # NEW — bata before herxu/sunxu (e.g. "Gujarat bata herxu regular")
    r'\bbata\b.*\b(herxu|sunxu)\b',
    # NEW — herxu followed by "regular" (e.g. "bata herxu regular news")
    r'\b(herxu|sunxu)\b.*\bregular\b',
    r'\bdainik\s+herne\b',
    r'\bregular\s+(listener|viewer|sunxu|herxu)\b',
    r'\bma\s+dainik\b',
    r'\bwatching\s+(from|live)\b',
    r'बाट\s*दैनिक\s*हेर्ने',
    r'नगरपालिका.*दैनिक',
    r'bata\s+dainik',
    r'\bरेगुलर\b',
    r'न\s*पा\s+बाट\s+रेगुलर',
    r'\bpa\s+bata\s+regular\b',
]

REACTION_PATTERNS = [
    r'^(hoho|haha|lol|wow|finally)\b',
    r'\bjel\s+(kochnu|haal|halnu)\b',
    r'^sabai\s+\w+\s+(lai\s+)?jel\b',
    r'^\w+\s+le\s+thik\s+bhan',
    r'^\w{1,15}\s+sir\s+please\b',
    r'kunai\s+pani\s+parti\s+ko.*baputi',
]

VAGUE_ENGLISH_PATTERNS = [
    r'^if\s+\w+\s+(come|win|lose|get|become)',
    r'^if\s+u\s+',
    r'^just\s+look\s+(at|in|what)',
    r'^(see|look|watch)\s+what\s+happen',
    r'baputi\s+banera\s+basdainan',
]

# NEW: Religious/devotional spam patterns
RELIGIOUS_SPAM_PATTERNS = [
    r'\bradhe\s+radhe\b',
    r'\bom\s+shanti\b',
    r'\bjay\s+shree\s+ram\b',
    r'\bjay\s+shri\s+ram\b',
    r'\bsadh\s+guru\b',
    r'\bhari\s+om\b',
    r'\bom\s+namah\b',
    r'\bbhagwan\s+jay\b',
    r'\bjay\s+mata\b',
    r'\bjai\s+shree\b',
]

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "]+", flags=re.UNICODE
)


# ---------------------------------------------------------------------------
# Main cleaner
# ---------------------------------------------------------------------------

def clean_youtube_chat(raw_lines, remove_all_duplicates=False, normalize_repeated_chars=False):
    cleaned_messages = []
    prev_msg = None
    seen = set()

    for line in raw_lines:
        line = str(line)

        # 1. Keyword filter (whole line)
        if any(re.search(kw, line, re.IGNORECASE) for kw in KEYWORDS_TO_REMOVE):
            continue

        # 2. HTML entities
        line = re.sub(r'&quot;', '', line)
        line = re.sub(r'&amp;', '&', line)
        line = re.sub(r'&lt;', '<', line)
        line = re.sub(r'&gt;', '>', line)
        line = re.sub(r'&nbsp;', ' ', line)
        line = re.sub(r'&#39;?', '', line)
        line = re.sub(r'&[a-zA-Z]+;', '', line)
        line = re.sub(r'&[^;\s]+;', '', line)

        # 3. Emojis
        line = EMOJI_PATTERN.sub('', line)

        # 4. All URLs and links
        line = re.sub(r'https?://\S+', '', line)
        line = re.sub(r'ftp://\S+', '', line)
        line = re.sub(r'www\.\S+', '', line)
        line = re.sub(r'youtu\.be/\S+', '', line)
        line = re.sub(r'youtube\.com/\S+', '', line)
        line = re.sub(r'\S+\.(com|org|net|io|co|in|ly|be)/\S+', '', line)
        line = re.sub(r'http\S+', '', line)

        # 5. HTML tags (proper and malformed)
        line = re.sub(r'<[^>]+>', '', line)
        line = re.sub(r'</?[a-zA-Z][^>]*>?', '', line)
        line = re.sub(r'\b(a\s+href|href\s*=)[^\s]*', '', line, flags=re.IGNORECASE)
        line = re.sub(r'href\s*=\s*["\'][^"\']*["\']', '', line)

        # 6. Leftover 'br' text from stripped <br> tags
        line = re.sub(r'\bbr\b', '', line)

        # 7. YouTube channel IDs (long alphanumeric strings 20+ chars)
        line = re.sub(r'\b[A-Za-z0-9_-]{20,}\b', '', line)

        # 8. Line numbers
        line = re.sub(r'^\s*\d+\s+', '', line)

        # 9. Timestamps
        line = re.sub(r'\b\d+:\d+(?::\d+)?\b', '', line)

        # 10. @mentions
        line = re.sub(r'@\S+', '', line)
        line = re.sub(r'@', '', line)

        # 11. Promotional phrases
        for phrase in PROMOTIONAL_PHRASES:
            line = re.sub(phrase, '', line, flags=re.IGNORECASE)

        # 12. Strip trailing laughter/spam
        line = re.sub(r'(\b(ha|haha|lol|xD)\s*){2,}$', '', line, flags=re.IGNORECASE)
        line = re.sub(r'(\bha\s+){3,}', '', line, flags=re.IGNORECASE)

        # 13. Standalone noise words
        for noise in [r'\balso\b', r'\bra\b', r'\bbr\b']:
            line = re.sub(noise, '', line, flags=re.IGNORECASE)

        # 14. Special characters (keep Devanagari + basic punctuation)
        line = re.sub(r'[^\w\s\u0900-\u097F.,!?;:()\-\'"]+', '', line)

        # 15. Normalize punctuation and whitespace
        line = re.sub(r'\.{2,}', '', line)
        line = re.sub(r'([!?])\1+', r'\1', line)
        line = re.sub(r'^[.\s]+|[.\s]+$', '', line)
        line = re.sub(r'\s+', ' ', line).strip()

        if not line:
            continue

        # 16. Optional: normalize repeated characters
        if normalize_repeated_chars:
            line = re.sub(r'(.)\1{2,}', r'\1', line)

        line = re.sub(r'\s+', ' ', line).strip()
        if not line:
            continue

        # --- Content quality filters ---
        if is_garbled_devanagari(line):         continue  # 17
        if is_garbled_romanized(line):          continue  # 18
        if has_midword_script_mixing(line):     continue  # 19 NEW
        if any(re.search(p, line, re.IGNORECASE) for p in FILLER_PATTERNS):        continue  # 20
        if any(re.search(p, line, re.IGNORECASE) for p in LISTENER_PATTERNS):      continue  # 21
        if any(re.search(p, line, re.IGNORECASE) for p in REACTION_PATTERNS):      continue  # 22
        if any(re.search(p, line, re.IGNORECASE) for p in VAGUE_ENGLISH_PATTERNS): continue  # 23
        if any(re.search(p, line, re.IGNORECASE) for p in RELIGIOUS_SPAM_PATTERNS): continue  # 24 NEW
        if is_rhetorical_filler(line):          continue  # 25
        if is_low_content_opinion(line):        continue  # 26
        if is_off_topic_english(line):          continue  # 27 NEW

        # 28. Minimum word count (higher bar for romanized lines)
        ascii_ratio = sum(1 for c in line if ord(c) < 128) / max(len(line), 1)
        min_words = 8 if ascii_ratio > 0.7 else 6
        if len(line.split()) < min_words:
            continue

        # 29. Duplicate / near-duplicate — bidirectional ratio, threshold 0.70
        if remove_all_duplicates:
            if line in seen or is_near_duplicate(line, cleaned_messages):
                continue
            seen.add(line)
        else:
            if line == prev_msg or is_near_duplicate(line, [prev_msg] if prev_msg else []):
                continue

        cleaned_messages.append(line)
        prev_msg = line

    return cleaned_messages


In [31]:
sample_chat = data[200:400]
cleaned_messages = clean_youtube_chat(sample_chat, remove_all_duplicates=False, normalize_repeated_chars=False)

In [32]:
cleaned_messages

['यो अप्रवासी हरु का समस्या बालेन र रबि को हजुर बा आए पनि समाधान गर्न सक्दैनन् ?',
 'Tehi sambidhan ma vote jitera kp lekhak jel halxu bannu 23 gatte school ko uniform ma ukasne maraune lai pani kanun lagdaina rabi balen le sambidhan vitrai bata vote leko ago laune rsp bidharthi marne balen pani batne sudan gurung lai kanun kagxa',
 'yo patrakar pagal ho yesle views ko lagi balen rabi ko half half karyakal vaneko xa',
 'प्र मन्त्री शुसिला कारकि जि तपाईं सबै काम राम्रो गरनु भाेसाथे भदाे२३,२४ गते काप्रतिबेदन खुलासा गरिरहे छेन किन पियम शुसिला कारकि जि जादा जादे याेपनि खुलासा गरेर जानुस कियसमा तपाईं काे रधनटि का लिडरहरु् छन ।',
 'शेखरकाेईराला केन्द्रीयसमितीकाे बैठकमा उपस्थीती हुन्छकी हुदैन टिकट पाउने बेला महिले मिलाउने काेसिस गरेकाे हु भन्थ्याे अब मिलाउछ कि मिलाउदैन हेर्न बाकिछ',
 'बिग्रेका नेता भनाउँदाहरूको बोली र भनाईलाई राखेर समाचार लम्ब्याउँने र धमिल्याउँने काम नगर्दा नै राम्रो हुन्छ सम्वाददाताहरू !',
 'रवि को धमिलो छवि हेरेर मुल्यांकन गरेर पनि गोप्य तवरले भन्नु सुझाव दिनु अगावै भित्री